In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created: January 2026
Author: Thomas Moerman
Description: Notebook for fine-tuning an LLM (TinyLlama) for machine translation using QLoRA.
             Self-contained notebook designed to run on Google Colab.
"""

# Fine-tuning an LLM for Machine Translation

This notebook walks you through **fine-tuning a pretrained LLM** (TinyLlama-1.1B-Chat) for machine translation using the 🤗 Hugging Face ecosystem and parameter-efficient fine-tuning (PEFT).

## What you'll learn
- How to download and prepare parallel translation data
- How to format data for instruction-tuned models using chat templates
- How to use **QLoRA** (Quantized Low-Rank Adaptation) for efficient fine-tuning
- How to train with the `SFTTrainer` from TRL
- How to run inference with your fine-tuned translation model

## Why QLoRA?
Fine-tuning even a 1B parameter model benefits from QLoRA, which combines:
- **4-bit quantization**: Reduces memory footprint dramatically
- **LoRA adapters**: Only trains a small number of additional parameters

This allows fine-tuning on free-tier Google Colab GPUs!

## 1) Imports + Environment Check

We use:
- **transformers**: models, tokenizers, and training utilities
- **datasets**: for loading and processing data
- **peft**: Parameter-Efficient Fine-Tuning (LoRA)
- **trl**: Transformer Reinforcement Learning (SFTTrainer)
- **bitsandbytes**: 4-bit quantization

We'll also set a random seed for reproducibility.


In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate

In [ ]:
import os
import json
import torch
import pandas as pd
from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    PeftConfig,
)

from trl import SFTTrainer, SFTConfig

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

set_seed(42)

## 2) Download the Translation Data

We'll use the English-French translation dataset from Hugging Face Hub (`LT3/nfr_bt_nmt_english-french`).

The dataset is loaded directly using the `datasets` library — no file downloads or uploads needed.

In [ ]:
# Configuration
REPO_NAME = "LT3/nfr_bt_nmt_english-french"

# Load the dataset directly from Hugging Face
print(f"Loading dataset from: {REPO_NAME}")
dataset = load_dataset(REPO_NAME)
print("Dataset loaded successfully!")
print(dataset)

## 3) Prepare the Data

We extract parallel sentence pairs directly from the loaded dataset.

For **instruction-tuned models**, we need to format the data as a conversation:
- **User**: Provides the source sentence with translation instruction
- **Assistant**: Provides the translation

For this tutorial, we'll use a small subset:
- **Train**: 2,000 examples
- **Validation**: 500 examples
- **Test**: 10 examples (for quick inference testing)

In [ ]:
# Extract parallel sentence pairs from the dataset
TRAIN_SAMPLES = 2000
VAL_SAMPLES = 500
TEST_SAMPLES = 10

en_train = dataset["train"]["english"][:TRAIN_SAMPLES]
fr_train = dataset["train"]["french"][:TRAIN_SAMPLES]

en_val = dataset["validation"]["english"][:VAL_SAMPLES]
fr_val = dataset["validation"]["french"][:VAL_SAMPLES]

en_test = dataset["test"]["english"][:TEST_SAMPLES]
fr_test = dataset["test"]["french"][:TEST_SAMPLES]

print(f"Dataset sizes:")
print(f"  Train: {len(en_train)} pairs")
print(f"  Validation: {len(en_val)} pairs")
print(f"  Test: {len(en_test)} pairs")

# Preview some examples
print("\n--- Sample training pairs ---")
for i in range(3):
    print(f"EN: {en_train[i]}")
    print(f"FR: {fr_train[i]}")
    print()

## 4) Load the Model and Tokenizer

We'll use **TinyLlama-1.1B-Chat-v1.0**, a compact but capable instruction-tuned model that fits easily on Colab GPUs.

### Key configurations:
- **4-bit quantization** (NF4): Dramatically reduces memory usage
- **Double quantization**: Further memory savings
- **bfloat16 compute**: For stable training

In [ ]:
# Model configuration
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "tinyllama-translation-en-fr"

In [ ]:
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading model: {MODEL_NAME}")
print("This may take a few minutes...")

# Load the model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    use_cache=False,  # Disable cache for training
)

print(f"Model loaded successfully!")
print(f"Model dtype: {model.dtype}")

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    add_bos_token=True,
    add_eos_token=False,  # SFTTrainer adds EOS when packing=True
)

# Set padding token (required for batched training)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Right padding for causal LM

print(f"Tokenizer loaded!")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"BOS token: {tokenizer.bos_token}")
print(f"EOS token: {tokenizer.eos_token}")
print(f"PAD token: {tokenizer.pad_token}")

## 5) Format Data with Chat Template

Instruction-tuned models expect data in a specific chat format. We use the tokenizer's `apply_chat_template()` method to format our translation examples correctly.

TinyLlama-Chat uses the Zephyr/ChatML format:
```
<|user|>
Translate this sentence from English to French:
English: <source sentence>
French:</s>
<|assistant|>
<target sentence></s>
```

In [8]:
def format_translation_example(en_sentence, fr_sentence=None, tokenizer=None, for_training=True):
    """
    Format a translation example using the model's chat template.
    
    Args:
        en_sentence: English source sentence
        fr_sentence: French target sentence (None for inference)
        tokenizer: The tokenizer with chat template
        for_training: If True, include the target translation
    
    Returns:
        Formatted string ready for the model
    """
    # Create the user message with translation instruction
    user_message = f"""Translate this sentence from English to French:
English: {en_sentence}
French:"""
    
    # Build the chat
    chat = [{"role": "user", "content": user_message}]
    
    # Add assistant response for training
    if for_training and fr_sentence:
        chat.append({"role": "assistant", "content": fr_sentence})
    
    # Apply the chat template
    formatted = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=not for_training  # Add prompt for inference
    )
    
    return formatted


def create_dataset(en_sentences, fr_sentences, tokenizer, for_training=True):
    """
    Create a HuggingFace Dataset from parallel sentences.
    """
    formatted_examples = []
    
    for en, fr in zip(en_sentences, fr_sentences):
        formatted = format_translation_example(
            en, fr, tokenizer, for_training=for_training
        )
        formatted_examples.append(formatted)
    
    return Dataset.from_dict({"text": formatted_examples})


# Create datasets
print("Creating training dataset...")
train_dataset = create_dataset(en_train, fr_train, tokenizer, for_training=True)

print("Creating validation dataset...")
val_dataset = create_dataset(en_val, fr_val, tokenizer, for_training=True)

print(f"\nDataset sizes:")
print(f"  Train: {len(train_dataset)}")
print(f"  Validation: {len(val_dataset)}")


Creating training dataset...
Creating validation dataset...

Dataset sizes:
  Train: 2000
  Validation: 500


In [9]:
# Preview a formatted training example
print("=" * 60)
print("FORMATTED TRAINING EXAMPLE:")
print("=" * 60)
print(train_dataset[0]["text"])
print("=" * 60)


FORMATTED TRAINING EXAMPLE:
<s>[INST] Translate this sentence from English to French:
English: Article 199b is replaced by the following:
French:[/INST] l'article 199 ter est remplacé par le texte suivant:</s>


## 6) Configure LoRA (Low-Rank Adaptation)

LoRA adds small trainable matrices to the model's attention layers. This allows us to:
- Train only ~0.5% of the parameters
- Keep the original model weights frozen
- Save only the small adapter weights

### Key LoRA hyperparameters:
- **r (rank)**: Size of the low-rank matrices (higher = more capacity, more memory)
- **lora_alpha**: Scaling factor for LoRA weights
- **lora_dropout**: Dropout for regularization

In [ ]:
# Configure LoRA
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,  # Rank of the update matrices
    bias="none",
    task_type="CAUSAL_LM",
    # Target the attention layers
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# Prepare model for k-bit training (enables gradient checkpointing, etc.)
model = prepare_model_for_kbit_training(model)

# IMPORTANT: Don't call get_peft_model() here!
# SFTTrainer will apply the peft_config when we pass it to the trainer.

print("LoRA configuration defined.")
print(f"  Rank (r): {peft_config.r}")
print(f"  Alpha: {peft_config.lora_alpha}")
print(f"  Target modules: {peft_config.target_modules}")
print("\nSFTTrainer will apply LoRA during initialization.")

## 7) Training Configuration

We use the `SFTTrainer` (Supervised Fine-Tuning Trainer) from TRL, which is designed for instruction tuning.

### Key training hyperparameters:
- **learning_rate**: 2e-4 is a good starting point for LoRA
- **batch_size**: Adjust based on your GPU memory
- **num_train_epochs**: 1-3 epochs usually sufficient
- **max_seq_length**: Maximum sequence length (including prompt + translation)
- **packing**: Combines short examples to maximize GPU utilization

In [ ]:
# Maximum sequence length for training
MAX_LENGTH = 512  # Increase if your translations are longer

# Training configuration using SFTConfig (combines TrainingArguments + SFT-specific settings)
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    
    # Training hyperparameters
    num_train_epochs=1,
    per_device_train_batch_size=4,  # Reduce if OOM
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 4 * 4 = 16
    
    # Optimizer settings
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    lr_scheduler_type="constant",
    
    # Precision - must match bnb_4bit_compute_dtype (bfloat16)
    bf16=True,
    
    # Logging and saving
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    
    # Other settings
    report_to="none",  # Change to "wandb" for experiment tracking
    save_total_limit=2,  # Keep only last 2 checkpoints
    
    # SFT-specific settings
    max_length=MAX_LENGTH,  # Maximum length of tokenized sequences
    packing=True,  # Pack multiple examples into one sequence for efficiency
    dataset_text_field="text",  # Column name containing the text data
)

print("Training configuration:")
print(f"  Epochs: {sft_config.num_train_epochs}")
print(f"  Batch size: {sft_config.per_device_train_batch_size}")
print(f"  Gradient accumulation: {sft_config.gradient_accumulation_steps}")
print(f"  Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Learning rate: {sft_config.learning_rate}")
print(f"  Max length: {sft_config.max_length}")
print(f"  Packing: {sft_config.packing}")

In [ ]:
# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print("Trainer initialized successfully!")

## 8) Train the Model

Now let's fine-tune! Training time depends on:
- Dataset size
- GPU speed
- Batch size

With TinyLlama (1.1B params), 2,000 examples, and 1 epoch, this should be quite fast on a Colab GPU.

In [ ]:
# Start training!
print("Starting training...")
print("=" * 60)

trainer.train()

print("=" * 60)
print("Training complete!")

In [ ]:
# Save the training logs
logs = trainer.state.log_history
logs_path = os.path.join(OUTPUT_DIR, "training_logs.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(logs_path, "w") as f:
    json.dump(logs, f, indent=2)

print(f"Training logs saved to: {logs_path}")

# Print final metrics
print("\nFinal training metrics:")
for log in logs:
    if "loss" in log:
        print(f"  Step {log.get('step', 'N/A')}: loss = {log['loss']:.4f}")

## 9) Save the Model

We save both:
- The LoRA adapter weights (small)
- The tokenizer

The adapter can be loaded on top of the base model for inference.

In [ ]:
# Save the fine-tuned model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to: {OUTPUT_DIR}")
print(f"\nSaved files:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}: {size / 1024 / 1024:.2f} MB")


## 10) Inference: Generate Translations

Now let's test our fine-tuned model! We'll use the model from memory to generate translations for test sentences.

In [ ]:
# For inference, we can reload the model fresh
# (or continue using the trained model from memory)

# Option 1: Use the model from memory (already loaded)
inference_model = model
inference_tokenizer = tokenizer

# Option 2: Load from disk (uncomment if needed)
# from peft import PeftModel, PeftConfig
# 
# # Load the PEFT config to get base model name
# peft_config = PeftConfig.from_pretrained(OUTPUT_DIR)
# 
# # Load base model
# base_model = AutoModelForCausalLM.from_pretrained(
#     peft_config.base_model_name_or_path,
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
# )
# 
# # Load the LoRA adapter
# inference_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
# inference_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

print("Model ready for inference!")


In [ ]:
def translate(english_text, model, tokenizer, max_new_tokens=100):
    """
    Translate English text to French using the fine-tuned model.
    
    Args:
        english_text: The English sentence to translate
        model: The fine-tuned model
        tokenizer: The tokenizer
        max_new_tokens: Maximum tokens to generate
    
    Returns:
        The French translation
    """
    # Format the prompt
    prompt = format_translation_example(
        english_text, None, tokenizer, for_training=False
    )
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Greedy decoding for deterministic output
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the generated part
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    translation = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return translation.strip()


In [ ]:
# Test on our held-out test examples
print("=" * 70)
print("TRANSLATION RESULTS ON TEST SET")
print("=" * 70)

for i, (en, fr_ref) in enumerate(zip(en_test, fr_test)):
    fr_pred = translate(en, inference_model, inference_tokenizer)
    
    print(f"\n--- Example {i+1} ---")
    print(f"English:    {en}")
    print(f"Reference:  {fr_ref}")
    print(f"Predicted:  {fr_pred}")


In [ ]:
# Try your own sentences!
custom_sentences = [
    "The weather is beautiful today.",
    "I love learning new languages.",
    "Machine translation has improved significantly in recent years.",
    "Can you help me find the train station?",
]

print("=" * 70)
print("CUSTOM TRANSLATIONS")
print("=" * 70)

for en in custom_sentences:
    fr = translate(en, inference_model, inference_tokenizer)
    print(f"\nEN: {en}")
    print(f"FR: {fr}")


## (Optional) Quick BLEU Evaluation

Here's how to compute BLEU score on your test set:


In [ ]:
# Optional: Compute BLEU score
# Uncomment and run if you have sacrebleu installed
# pip install sacrebleu

# import sacrebleu
# 
# # Generate translations for test set
# predictions = [translate(en, inference_model, inference_tokenizer) for en in en_test]
# 
# # Compute BLEU
# bleu = sacrebleu.corpus_bleu(predictions, [fr_test])
# print(f"BLEU score: {bleu.score:.2f}")
